# This Notebook Verify's and Explores the structure of the 3D array/Tensor for partition 1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Verify Structure

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)

In [ ]:
!pip install lets-plot
import sys
# Ensure the newly installed package is in the path
from lets_plot import *
LetsPlot.setup_html()

In [ ]:
from tqdm.auto import tqdm

In [ ]:
import numpy as np
import pandas as pd
import json

# Load the file using the provided path
file_path = '/content/drive/MyDrive/solar_flare_forecasting/processed/partition1_combined.npz'
data = np.load(file_path, allow_pickle=True)

print(f"Keys in data: {data.files}")

# Extracting arrays
X = data["X"]
y_text = data["fl_nf_label"]
flare_class = data["flare_class"]
harpnum = data["harpnum"]
source_file = data["source_file"]
n_interpolated_rows = data["n_interpolated_rows"]
xrquality_degraded = data["xrquality_degraded"]

# Display summary statistics
print("X shape:", X.shape)
print("Labels distribution:\n", pd.Series(y_text).value_counts())
print("Flare classes distribution:\n", pd.Series(flare_class).value_counts())
print("Unique HARPNUMs:", len(np.unique(harpnum)))
print("Total NaNs in X:", np.isnan(X).sum())

In [ ]:
# Display the main data array X
print("Displaying the data array X:")
X

In [ ]:
import json

# Try to load feature names to use as column headers
feature_json_path = '/content/drive/MyDrive/solar_flare_forecasting/processed/partition1_feature_columns.json'
try:
    with open(feature_json_path, 'r') as f:
        feature_columns = json.load(f)
except Exception:
    feature_columns = [f'feature_{i}' for i in range(X.shape[2])]

# Convert the first sample (index 0) to a DataFrame
# X[0] has shape (60, 47) -> (time_steps, features)
sample_df = pd.DataFrame(X[0], columns=feature_columns)

print("Displaying the first sample (60 time steps) as a DataFrame:")
display(sample_df)

In [ ]:
import json
import pandas as pd

# Path to the feature columns JSON
feature_json_path = '/content/drive/MyDrive/solar_flare_forecasting/processed/partition1_feature_columns.json'

try:
    with open(feature_json_path, 'r') as f:
        feature_columns = json.load(f)
    print("Successfully loaded feature names.")
except Exception as e:
    print(f"Could not load feature names, using defaults. Error: {e}")
    feature_columns = [f'feature_{i}' for i in range(X.shape[2])]

# Convert the first sample (index 0) to a DataFrame
# X[0] shape is (60, 47), representing 60 time steps and 47 features
sample_df = pd.DataFrame(X[0], columns=feature_columns)

print("Displaying the first sample (60 time steps) as a DataFrame:")
display(sample_df)

In [ ]:
import matplotlib.pyplot as plt

# Let's pick a feature to visualize over the 60 timesteps
# Using 'TOTUSJH' (Total unsigned vertical current helicity) as an example
feature_to_plot = 'TOTUSJH'
sample_index = 0

plt.figure(figsize=(10, 5))
plt.plot(sample_df[feature_to_plot], marker='o', linestyle='-')
plt.title(f"Feature '{feature_to_plot}' over 60 Timesteps (Sample {sample_index})")
plt.xlabel("Timestep (Index)")
plt.ylabel("Value")
plt.grid(True)
plt.show()

print(f"The DataFrame rows 0-59 correspond exactly to the sequence of time for sample {sample_index}.")

In [ ]:
# Check if any column name contains 'time' or 'date'
time_related_cols = [col for col in feature_columns if 'TIME' in col.upper() or 'DATE' in col.upper()]

if time_related_cols:
    print(f"Found time-related columns: {time_related_cols}")
    display(sample_df[time_related_cols].head())
else:
    print("No explicit timestamp column found in the features. Time is represented by the row index (0-59).")

# Also check the first few column names to be sure
print("\nFirst 10 columns:", feature_columns[:10])

In [ ]:
import os
import pandas as pd

files_path = '/content/drive/MyDrive/solar_flare_forecasting/processed/partition1_metadata.csv'
# Check if the file is empty before reading
df_metadata = pd.read_csv(files_path)

df_metadata


### Files that had values interpolated

In [ ]:
# Filter metadata to find rows where at least one row was interpolated
interpolated_rows = df_metadata[df_metadata['n_interpolated_rows'] > 0]

print(f"Number of samples with interpolation: {len(interpolated_rows)}")
interpolated_rows.head(20)

### Are rows interpolated in the middle or edge rows?

In [ ]:
import numpy as np

# Identify indices of all rows with interpolation
interp_indices = df_metadata[df_metadata['n_interpolated_rows'] > 0].index.tolist()

middle_count = 0
boundary_count = 0

# The 'was_interpolated' feature is the last column (index 46)
WAS_INTERP_IDX = 46

# Using tqdm to track progress through the interpolated samples
for idx in tqdm(interp_indices, desc="Checking interpolation positions"):
    # Get the sequence of interpolation flags (60 timesteps)
    interp_flags = X[idx, :, WAS_INTERP_IDX]

    # Check boundaries: index 0 or index 59
    has_boundary = (interp_flags[0] == 1) or (interp_flags[59] == 1)

    # Check middle: indices 1 through 58
    has_middle = np.any(interp_flags[1:59] == 1)

    if has_boundary:
        boundary_count += 1
    if has_middle:
        middle_count += 1

print(f"Total samples with interpolation: {len(interp_indices)}")
print(f"Samples with interpolation in middle rows (1-58): {middle_count}")
print(f"Samples with interpolation in boundary rows (0 or 59): {boundary_count}")
print('\nNote: A single file can have both middle and boundary interpolation.')

In [ ]:
# Identify indices of rows with boundary interpolation based on the previous calculation
boundary_interp_indices = []
WAS_INTERP_IDX = 46

interp_indices = df_metadata[df_metadata['n_interpolated_rows'] > 0].index.tolist()

for idx in interp_indices:
    interp_flags = X[idx, :, WAS_INTERP_IDX]
    if (interp_flags[0] == 1) or (interp_flags[59] == 1):
        boundary_interp_indices.append(idx)

# Create a DataFrame showing the files with boundary interpolation
boundary_files_df = df_metadata.iloc[boundary_interp_indices][['source_file', 'n_interpolated_rows', 'xrquality_degraded']]

print(f"Total files with boundary interpolation: {len(boundary_files_df)}")
display(boundary_files_df.head(20))

In [ ]:
# Count how many files in boundary_files_df have 5 or more interpolated rows
five_plus_interp = boundary_files_df[boundary_files_df['n_interpolated_rows'] >= 5]

print(f"Number of boundary files with 5+ interpolated values: {len(five_plus_interp)}")
print("\nFirst 10 such files:")
display(five_plus_interp.head(10))

In [ ]:
row_index = 3569
target_filename = df_metadata.iloc[row_index]['source_file']

try:
    # Find index in source_file array
    idx = np.where(source_file == target_filename)[0][0]

    # Extract data for this sample
    file_df_3569 = pd.DataFrame(X[idx], columns=feature_columns)

    print(f"Visualizing USFLUX for Index {row_index} (Filename: {target_filename})")

    # Plot using lets-plot
    from lets_plot import *
    LetsPlot.setup_html()

    p = ggplot(file_df_3569.reset_index()) + \
        geom_point(aes(x='index', y='USFLUX'), color='blue', size=3) + \
        ggtitle(f'USFLUX Scatter Plot - Row Index {row_index}') + \
        xlab('Timestep (Index)') + ylab('USFLUX Value')

    p.show()
except Exception as e:
    print(f"Error: {e}")

## Visulizing Interpolated values

### Value in the middle rows.

In [ ]:
row_index = 254
target_filename_264 = df_metadata.iloc[row_index]['source_file']

try:
    # Find index in source_file array
    idx = np.where(source_file == target_filename_264)[0][0]

    # Extract and display data from X
    file_df = pd.DataFrame(X[idx], columns=feature_columns)

    print(f"Data for Index {row_index} (Filename: {target_filename_264})")
    display(file_df)
except Exception as e:
    print(f"Error: {e}")

In [ ]:
from lets_plot import *
LetsPlot.setup_html()

# Create a scatter plot for USFLUX using lets-plot
# file_df corresponds to row index 254 as per the previous steps
p = ggplot(file_df.reset_index()) + \
    geom_point(aes(x='index', y='USFLUX'), color='orange', size=3) + \
    ggtitle(f'USFLUX Scatter Plot (Lets-Plot) - Row Index 254') + \
    xlab('Timestep (Index)') + ylab('USFLUX Value')

p.show()

### Value in end rows


In [ ]:
row_index = 19
target_filename = df_metadata.iloc[row_index]['source_file']

try:
    # Find index in source_file array
    idx = np.where(source_file == target_filename)[0][0]

    # Extract and display data from X
    file_df = pd.DataFrame(X[idx], columns=feature_columns)

    print(f"Data for Index {row_index} (Filename: {target_filename_264})")
    display(file_df)
except Exception as e:
    print(f"Error: {e}")

In [ ]:
from lets_plot import *
LetsPlot.setup_html()

# Create a scatter plot for USFLUX using lets-plot
# file_df corresponds to row index 254 as per the previous steps
p = ggplot(file_df.reset_index()) + \
    geom_point(aes(x='index', y='USFLUX'), color='orange', size=3) + \
    ggtitle(f'USFLUX Scatter Plot (Lets-Plot) - Row Index 254') + \
    xlab('Timestep (Index)') + ylab('USFLUX Value')

p.show()

## another

In [ ]:
row_index = 30
target_filename = df_metadata.iloc[row_index]['source_file']

try:
    # Find index in source_file array
    idx = np.where(source_file == target_filename)[0][0]

    # Extract and display data from X
    file_df = pd.DataFrame(X[idx], columns=feature_columns)

    print(f"Data for Index {row_index} (Filename: {target_filename_264})")
    display(file_df)
except Exception as e:
    print(f"Error: {e}")

In [ ]:
from lets_plot import *
LetsPlot.setup_html()

# Create a scatter plot for USFLUX using lets-plot
# file_df corresponds to row index 254 as per the previous steps
p = ggplot(file_df.reset_index()) + \
    geom_point(aes(x='index', y='USFLUX'), color='orange', size=3) + \
    ggtitle(f'USFLUX Scatter Plot (Lets-Plot) - Row Index 254') + \
    xlab('Timestep (Index)') + ylab('USFLUX Value')

p.show()

## How many files have 5+ rows interpolated?

In [ ]:
# Calculate how many files in the entire dataset have 5 or more interpolated rows
total_five_plus = df_metadata[df_metadata['n_interpolated_rows'] >= 5]

print(f"Total files in the 3D array with 5+ interpolated rows: {len(total_five_plus)}")

In [ ]:
# Show the files that have 5 or more interpolated rows
# We already calculated total_five_plus in the previous cell
print(f"Displaying the first 20 out of {len(total_five_plus)} files with 5+ interpolated rows:")
display(total_five_plus[['source_file', 'n_interpolated_rows', 'flare_class']].head(20))

In [ ]:
from lets_plot import *
LetsPlot.setup_html()

# Create a bar plot of the n_interpolated_rows distribution
p = ggplot(total_five_plus) + \
    geom_bar(aes(x='n_interpolated_rows'), fill='#3498db') + \
    ggtitle('Distribution of Interpolated Rows per File') + \
    xlab('Number of Interpolated Rows') + \
    ylab('Count of Files') + \
    scale_x_continuous(breaks=list(range(0, int(df_metadata['n_interpolated_rows'].max()) + 1))) + \
    theme_minimal()

p.show()

In [ ]:
# Get value counts of n_interpolated_rows for the 5+ subset
print("Value counts for n_interpolated_rows (5 or more):")
print(total_five_plus['n_interpolated_rows'].value_counts().sort_index(ascending=False))